In [9]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv

load_dotenv()
model=init_chat_model(
    model="gpt-4.1-mini",
    model_provider="openai"
)
response=model.invoke("你有embedding类型的模型嘛")
print(response.content)


我本身是基于大型语言模型（LLM）架构构建的，能够生成文本嵌入（embedding），但我并不是专门的embedding模型。不过，我可以帮助你生成文本的向量表示，用于语义搜索、相似度计算等任务。

如果你需要专门的embedding模型，OpenAI 提供了专门的 Embedding API，比如 `text-embedding-ada-002`，它是一个专门用于生成高质量文本向量的模型。

你可以告诉我你的具体需求，比如你想生成哪种文本的embedding，是句子、段落还是其他内容，我可以帮你生成或指导你如何调用相关的embedding接口。


In [4]:
from langgraph.graph.message import REMOVE_ALL_MESSAGES
from langgraph.runtime import Runtime
from typing import Any
from langchain.agents.middleware import before_model
from langchain_core.messages import HumanMessage, RemoveMessage
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents import create_agent, AgentState


@before_model
def trim_message(state:AgentState,runtime:Runtime)->dict[str,Any]|None:
     messages=state["messages"]
     if messages <=3:
         return None
     first_message=messages[0]
     #如果有偶数条消息，则取最近的3条消息，则去最近的4条消息
     recent_messages=messages[-3:] if len(messages)%2==0 else messages[-4:]
     new_messages=[first_message]+recent_messages
     return {"messages":[
         RemoveMessage(id=REMOVE_ALL_MESSAGES),*new_messages
     ]}
agent=create_agent(
    model=model,
    checkpointer=InMemorySaver()
)
config={"configurable":{"thread_id":"1"}}
agent.invoke({"messages":[HumanMessage("你好，我是黄鑫远")]},config=config)
agent.invoke({"messages":[HumanMessage("从现在开始你叫小王吧")]},config=config)
agent.invoke({"messages":[HumanMessage("今天天气不错")]},config=config)
final_response=agent.invoke({"messages":[HumanMessage("告诉我,你是是谁？我是谁？")]},config=config)
for msg in final_response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

你好，我是黄鑫远
================================== Ai Message ==================================

你好，黄鑫远！我是DeepSeek助手，很高兴认识你。有什么我可以帮你的吗？无论是问题解答、信息查询，还是聊天陪伴，我都在这里。😊
================================ Human Message =================================

从现在开始你叫小王吧
================================== Ai Message ==================================

好的，从现在开始我就叫“小王”啦！😊 有什么需要尽管吩咐，黄鑫远，很高兴为你服务！
================================ Human Message =================================

今天天气不错
================================== Ai Message ==================================

是啊，今天天气确实不错！☀️ 阳光暖暖的，看着就让人心情好。你是出门逛了，还是正打算去晒晒太阳？不管怎样，这样的日子适合做点开心的事～
================================ Human Message =================================

告诉我,你是是谁？我是谁？
================================== Ai Message ==================================

哈哈，这个问题挺有哲学味的！🤔 不过认真回答你：

**我是谁？**  
我是小王——你的AI助手，由深度求索公司创造，目前正以“小王”的身份陪你聊天，帮你解答问题、出主意，或者单纯听你说说话，随时随地都在。

**你是谁？**  
你是我对话另一端的朋

In [ ]:
#裁剪策略
#用消息中间件summarization组件更好
